In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!tar -xzf "/content/drive/MyDrive/CodecRobust/processed_upload.tar.gz" -C /content/
print("Done.")
!ls /content/uncompressed/dev/ | head -3

Done.
LA_D_1006568.pt
LA_D_1008730.pt
LA_D_1010295.pt


In [4]:
!git clone https://github.com/roh1thbharathi/applied-dl-project-2.git
%cd /content/applied-dl-project-2
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, '/content/applied-dl-project-2/src')
print("Done.")

Cloning into 'applied-dl-project-2'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 72 (delta 22), reused 57 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 72.18 KiB | 2.78 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/applied-dl-project-2
Done.


In [ ]:
import shutil, os

os.makedirs("/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols", exist_ok=True)

for fname in ['ASVspoof2019.LA.cm.train.trn.txt',
              'ASVspoof2019.LA.cm.dev.trl.txt',
              'ASVspoof2019.LA.cm.eval.trl.txt']:
    shutil.copy(
        f"/content/drive/MyDrive/CodecRobust/protocols/LA/ASVspoof2019_LA_cm_protocols/{fname}",
        f"/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/{fname}"
    )

print("Done.")

# Training

In [3]:
import os
from pathlib import Path
import data_utils

original_init = data_utils.ASVspoof2019Preprocessed.__init__

def patched_init(self, asv_root, processed_root, split="train", contrastive=False):
    original_init(self, asv_root, processed_root, split, contrastive)
    available = set(
        f.replace('.pt', '')
        for f in os.listdir(Path(processed_root) / 'uncompressed' / split)
    )
    before = len(self.records)
    self.records = self.records[self.records['filename'].isin(available)].reset_index(drop=True)
    print(f"  [Filter] {split}: {before} → {len(self.records)} (matched preprocessed files)")

data_utils.ASVspoof2019Preprocessed.__init__ = patched_init
print("Patch applied.")

Patch applied.


In [9]:
ASV_ROOT       = "/content/asvspoof"
PROCESSED_ROOT = "/content"
print("Paths set.")

Paths set.


In [8]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving ASVspoof2019.LA.cm.train.trn.txt to ASVspoof2019.LA.cm.train.trn.txt
Saving ASVspoof2019.LA.cm.eval.trl.txt to ASVspoof2019.LA.cm.eval.trl.txt
Saving ASVspoof2019.LA.cm.dev.trl.txt to ASVspoof2019.LA.cm.dev.trl.txt
Uploaded: ['ASVspoof2019.LA.cm.train.trn.txt', 'ASVspoof2019.LA.cm.eval.trl.txt', 'ASVspoof2019.LA.cm.dev.trl.txt']


In [10]:
import os, shutil

os.makedirs("/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols", exist_ok=True)
os.makedirs("/content/drive/MyDrive/CodecRobust/protocols/LA/ASVspoof2019_LA_cm_protocols", exist_ok=True)

for fname in ['ASVspoof2019.LA.cm.train.trn.txt',
              'ASVspoof2019.LA.cm.dev.trl.txt',
              'ASVspoof2019.LA.cm.eval.trl.txt']:
    src   = f"/content/applied-dl-project-2/{fname}"
    local = f"/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/{fname}"
    drive = f"/content/drive/MyDrive/CodecRobust/protocols/LA/ASVspoof2019_LA_cm_protocols/{fname}"
    shutil.move(src, local)
    shutil.copy(local, drive)

print("Done.")
!ls /content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/

Done.
ASVspoof2019.LA.cm.dev.trl.txt	 ASVspoof2019.LA.cm.train.trn.txt
ASVspoof2019.LA.cm.eval.trl.txt


In [11]:
import shutil

# Backup original
shutil.copy('/content/applied-dl-project-2/src/model.py',
            '/content/applied-dl-project-2/src/model_backup.py')

# Read current file
with open('/content/applied-dl-project-2/src/model.py', 'r') as f:
    content = f.read()

# Make the 3 targeted changes
content = content.replace(
    "    def __init__(self, embed_dim=256, sinc_ch=70, sample_rate=16000, n_attn_heads=4):\n        super().__init__()\n        self.sinc",
    "    def __init__(self, embed_dim=256, sinc_ch=70, sample_rate=16000, n_attn_heads=4, use_temporal_attn=True):\n        super().__init__()\n        self.use_temporal_attn = use_temporal_attn\n        self.sinc"
)

content = content.replace(
    "        r    = self.temporal_attn(r)      # (B, 32, 128) — codec-robust focus",
    "        if self.use_temporal_attn:\n            r = self.temporal_attn(r)"
)

content = content.replace(
    "    def __init__(self, embed_dim=256, n_codec_classes=10, sample_rate=16000, n_attn_heads=4):\n        super().__init__()\n        # TemporalAttention is now INSIDE AASISTEncoder (on 32 nodes before pool)\n        self.encoder = AASISTEncoder(embed_dim=embed_dim, sample_rate=sample_rate,\n                                     n_attn_heads=n_attn_heads)",
    "    def __init__(self, embed_dim=256, n_codec_classes=10, sample_rate=16000, n_attn_heads=4, use_temporal_attn=True):\n        super().__init__()\n        self.encoder = AASISTEncoder(embed_dim=embed_dim, sample_rate=sample_rate,\n                                     n_attn_heads=n_attn_heads, use_temporal_attn=use_temporal_attn)"
)

with open('/content/applied-dl-project-2/src/model.py', 'w') as f:
    f.write(content)

# Reload
import sys
for mod in list(sys.modules.keys()):
    if mod in ['model', 'data_utils', 'evaluate']:
        del sys.modules[mod]

from model import CodecRobustDetector
m = CodecRobustDetector(use_temporal_attn=False)
print("Success. Params:", sum(p.numel() for p in m.parameters()))

Success. Params: 550948


In [14]:
import sys, os, importlib
from pathlib import Path

for mod in list(sys.modules.keys()):
    if mod in ['model', 'data_utils', 'evaluate']:
        del sys.modules[mod]

import data_utils
importlib.reload(data_utils)

_original_init = data_utils.ASVspoof2019Preprocessed.__init__

def patched_init(self, asv_root, processed_root, split="train", contrastive=False):
    _original_init(self, asv_root, processed_root, split, contrastive)
    available = set(f.replace('.pt','') for f in os.listdir(Path(processed_root)/'uncompressed'/split))
    before = len(self.records)
    self.records = self.records[self.records['filename'].isin(available)].reset_index(drop=True)
    print(f"  [Filter] {split}: {before} → {len(self.records)} (matched preprocessed files)")

data_utils.ASVspoof2019Preprocessed.__init__ = patched_init
print("Ready.")

Ready.


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
from pathlib import Path

from model import CodecRobustDetector, ContrastiveLoss
from data_utils import get_dataloaders, N_CODEC_CLASSES
from evaluate import compute_eer, compute_auc

# ── Config ────────────────────────────────────────────────
ALPHA               = 0.5    # GRL on
BETA                = 0.1    # Contrastive on
USE_CONTRASTIVE     = True
USE_TEMPORAL_ATTN   = False  # temporal off
EPOCHS              = 30
BATCH_SIZE          = 16
LR                  = 3e-4
MAX_SAMPLES         = 3000
EMBED_DIM           = 256
SAVE_EVERY          = 1
RESULTS_DIR         = "/content/drive/MyDrive/CodecRobust/results/grl_contrastive_no_temporal"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

os.makedirs(RESULTS_DIR, exist_ok=True)

loaders = get_dataloaders(
    asv_root=ASV_ROOT,
    preprocessed_root=PROCESSED_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    contrastive=USE_CONTRASTIVE,
    max_samples=MAX_SAMPLES,
)

model      = CodecRobustDetector(embed_dim=EMBED_DIM, n_codec_classes=N_CODEC_CLASSES,
                                  use_temporal_attn=USE_TEMPORAL_ATTN).to(device)
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)
df_crit    = nn.CrossEntropyLoss()
codec_crit = nn.CrossEntropyLoss()
con_crit   = ContrastiveLoss(temperature=0.07)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

out_dir     = Path(RESULTS_DIR)
start_epoch = 1
best_eer    = 1.0

latest_ckpt = sorted(out_dir.glob("checkpoint_epoch_*.pt"))
if latest_ckpt:
    ckpt = torch.load(latest_ckpt[-1], map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_eer    = ckpt['best_eer']
    print(f"Resumed from epoch {ckpt['epoch']}, best EER: {best_eer*100:.2f}%")
else:
    print("Starting fresh.")

def grl_lambda(step, total_steps, lam_max=1.0):
    p = step / max(total_steps, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)

total_steps = EPOCHS * len(loaders["train"])
step = (start_epoch - 1) * len(loaders["train"])

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for batch in loaders["train"]:
        lam = grl_lambda(step, total_steps)
        model.set_lambda(lam)
        wf        = batch["waveform"].to(device)
        labels    = batch["label"].to(device)
        codec_idx = batch["codec_idx"].to(device)
        out       = model(wf)
        loss      = df_crit(out["deepfake_logits"], labels) + ALPHA * codec_crit(out["codec_logits"], codec_idx)

        if USE_CONTRASTIVE and "waveform2" in batch:
            wf2   = batch["waveform2"].to(device)
            out2  = model(wf2)
            B     = wf.size(0)
            pairs = torch.stack([out["embedding"], out2["embedding"]], dim=1).view(2*B, -1)
            loss  = loss + BETA * con_crit(pairs)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        step += 1
    scheduler.step()

    model.eval()
    scores, labs = [], []
    with torch.no_grad():
        for batch in loaders["dev"]:
            s = torch.softmax(model(batch["waveform"].to(device))["deepfake_logits"], -1)[:, 1]
            scores.append(s.cpu())
            labs.append(batch["label"])
    scores = torch.cat(scores).numpy()
    labs   = torch.cat(labs).numpy()
    eer    = compute_eer(labs, scores)
    auc    = compute_auc(labs, scores)

    print(f"Epoch {epoch:02d}/{EPOCHS} | EER={eer*100:.2f}% AUC={auc:.4f} | time={time.time()-t0:.1f}s")

    if eer < best_eer:
        best_eer = eer
        torch.save(model.state_dict(), out_dir / "best_model.pt")
        print(f"  ✓ Best EER: {best_eer*100:.2f}% — saved")

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'eer': eer,
            'best_eer': best_eer,
        }, out_dir / f"checkpoint_epoch_{epoch:02d}.pt")
        print(f"  ✓ Checkpoint saved: epoch {epoch}")

print(f"\nDone. Best EER: {best_eer*100:.2f}%")

Device: cuda
Free GPU: 15.52 GB
  [DataLoader] Using preprocessed data from: /content
  [Filter] train: 25380 → 3000 (matched preprocessed files)
  [DataLoader] train: 3,000 samples | 10 codec variants available
  [Filter] dev: 24844 → 3000 (matched preprocessed files)
  [DataLoader] dev  : 3,000 samples | 10 codec variants available
  [DataLoader] WARN: could not load eval: No preprocessed folders found under /content. Run: python src/preprocess_codecs.py --asv_root <root>
Params: 550,948
Starting fresh.
Epoch 01/30 | EER=8.23% AUC=0.9710 | time=252.9s
  ✓ Best EER: 8.23% — saved
  ✓ Checkpoint saved: epoch 1
Epoch 02/30 | EER=7.72% AUC=0.9771 | time=255.8s
  ✓ Best EER: 7.72% — saved
  ✓ Checkpoint saved: epoch 2
Epoch 03/30 | EER=4.01% AUC=0.9901 | time=254.8s
  ✓ Best EER: 4.01% — saved
  ✓ Checkpoint saved: epoch 3
Epoch 04/30 | EER=33.88% AUC=0.7426 | time=255.1s
  ✓ Checkpoint saved: epoch 4
Epoch 05/30 | EER=13.00% AUC=0.9241 | time=254.6s
  ✓ Checkpoint saved: epoch 5
Epoch 06

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
from pathlib import Path

from model import CodecRobustDetector, ContrastiveLoss
from data_utils import get_dataloaders, N_CODEC_CLASSES
from evaluate import compute_eer, compute_auc

# ── Config ────────────────────────────────────────────────
ALPHA               = 0.0    # GRL off
BETA                = 0.0    # Contrastive off
USE_CONTRASTIVE     = False
USE_TEMPORAL_ATTN   = False  # temporal off
EPOCHS              = 30
BATCH_SIZE          = 32
LR                  = 3e-4
MAX_SAMPLES         = 3000
EMBED_DIM           = 256
SAVE_EVERY          = 1
RESULTS_DIR         = "/content/drive/MyDrive/CodecRobust/results/vanilla"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

loaders = get_dataloaders(
    asv_root=ASV_ROOT,
    preprocessed_root=PROCESSED_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    contrastive=USE_CONTRASTIVE,
    max_samples=MAX_SAMPLES,
)

model      = CodecRobustDetector(embed_dim=EMBED_DIM, n_codec_classes=N_CODEC_CLASSES,
                                  use_temporal_attn=USE_TEMPORAL_ATTN).to(device)
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)
df_crit    = nn.CrossEntropyLoss()
codec_crit = nn.CrossEntropyLoss()
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

out_dir     = Path(RESULTS_DIR)
start_epoch = 1
best_eer    = 1.0

latest_ckpt = sorted(out_dir.glob("checkpoint_epoch_*.pt"))
if latest_ckpt:
    ckpt = torch.load(latest_ckpt[-1], map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_eer    = ckpt['best_eer']
    print(f"Resumed from epoch {ckpt['epoch']}, best EER: {best_eer*100:.2f}%")
else:
    print("Starting fresh.")

def grl_lambda(step, total_steps, lam_max=1.0):
    p = step / max(total_steps, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)

total_steps = EPOCHS * len(loaders["train"])
step = (start_epoch - 1) * len(loaders["train"])

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for batch in loaders["train"]:
        lam = grl_lambda(step, total_steps)
        model.set_lambda(lam)
        wf        = batch["waveform"].to(device)
        labels    = batch["label"].to(device)
        codec_idx = batch["codec_idx"].to(device)
        out       = model(wf)
        loss      = df_crit(out["deepfake_logits"], labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        step += 1
    scheduler.step()

    model.eval()
    scores, labs = [], []
    with torch.no_grad():
        for batch in loaders["dev"]:
            s = torch.softmax(model(batch["waveform"].to(device))["deepfake_logits"], -1)[:, 1]
            scores.append(s.cpu())
            labs.append(batch["label"])
    scores = torch.cat(scores).numpy()
    labs   = torch.cat(labs).numpy()
    eer    = compute_eer(labs, scores)
    auc    = compute_auc(labs, scores)

    print(f"Epoch {epoch:02d}/{EPOCHS} | EER={eer*100:.2f}% AUC={auc:.4f} | time={time.time()-t0:.1f}s")

    if eer < best_eer:
        best_eer = eer
        torch.save(model.state_dict(), out_dir / "best_model.pt")
        print(f"  ✓ Best EER: {best_eer*100:.2f}% — saved")

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'eer': eer,
            'best_eer': best_eer,
        }, out_dir / f"checkpoint_epoch_{epoch:02d}.pt")
        print(f"  ✓ Checkpoint saved: epoch {epoch}")

print(f"\nDone. Best EER: {best_eer*100:.2f}%")

Device: cuda
Free GPU: 4.44 GB
  [DataLoader] Using preprocessed data from: /content
  [Filter] train: 25380 → 3000 (matched preprocessed files)
  [DataLoader] train: 3,000 samples | 10 codec variants available
  [Filter] dev: 24844 → 3000 (matched preprocessed files)
  [DataLoader] dev  : 3,000 samples | 10 codec variants available
  [DataLoader] WARN: could not load eval: No preprocessed folders found under /content. Run: python src/preprocess_codecs.py --asv_root <root>
Params: 550,948
Starting fresh.
Epoch 01/30 | EER=9.11% AUC=0.9572 | time=142.3s
  ✓ Best EER: 9.11% — saved
  ✓ Checkpoint saved: epoch 1
Epoch 02/30 | EER=8.06% AUC=0.9762 | time=142.4s
  ✓ Best EER: 8.06% — saved
  ✓ Checkpoint saved: epoch 2
Epoch 03/30 | EER=9.69% AUC=0.9693 | time=142.4s
  ✓ Checkpoint saved: epoch 3
Epoch 04/30 | EER=5.15% AUC=0.9884 | time=142.7s
  ✓ Best EER: 5.15% — saved
  ✓ Checkpoint saved: epoch 4
Epoch 05/30 | EER=8.65% AUC=0.9523 | time=142.3s
  ✓ Checkpoint saved: epoch 5
Epoch 06/30

# Experiments

In [6]:
# Cell 5: Set paths
ASV_ROOT       = "/content/asvspoof"
PROCESSED_ROOT = "/content"
print("Paths set.")

Paths set.


In [7]:
import os

checkpoint_map = {
    "1. Vanilla":          "/content/drive/MyDrive/CodecRobust/results/vanilla/best_model.pt",
    "2. Temporal only":    "/content/drive/MyDrive/Applied-DL-2/trained_models/base_best_model.pt",
    "3. GRL+Contrastive":  "/content/drive/MyDrive/CodecRobust/results/grl_contrastive_no_temporal/best_model.pt",
    "4. Full":             "/content/drive/MyDrive/Applied-DL-2/trained_models/full_best_model.pt",
}

for name, path in checkpoint_map.items():
    exists = os.path.exists(path)
    print(f"{name}: {'✓' if exists else '✗ MISSING'} — {path}")

1. Vanilla: ✓ — /content/drive/MyDrive/CodecRobust/results/vanilla/best_model.pt
2. Temporal only: ✓ — /content/drive/MyDrive/Applied-DL-2/trained_models/base_best_model.pt
3. GRL+Contrastive: ✓ — /content/drive/MyDrive/CodecRobust/results/grl_contrastive_no_temporal/best_model.pt
4. Full: ✓ — /content/drive/MyDrive/Applied-DL-2/trained_models/full_best_model.pt


In [9]:
def apply_patches():
    import sys, importlib, torch

    for mod in list(sys.modules.keys()):
        if mod in ['model', 'data_utils', 'evaluate']:
            del sys.modules[mod]

    import model as m

    # Patch AASISTEncoder.forward
    def new_encoder_forward(self, waveform):
        import torch.nn.functional as F
        x = waveform.unsqueeze(1)
        x = torch.abs(self.sinc(x))
        x = F.gelu(self.bn_sinc(x))
        skip = self.proj_skip(x)
        r = F.gelu(self.res1(x) + skip)
        r = F.gelu(self.res2(r) + r)
        r = F.gelu(self.res3(r) + r)
        r = self.pool(r)
        r = r.permute(0, 2, 1)
        r = self.gat1(r)
        r = self.gat2(r)
        if getattr(self, 'use_temporal_attn', True):
            r = self.temporal_attn(r)
        r = r.mean(dim=1)
        return self.head(r)

    m.AASISTEncoder.forward = new_encoder_forward

    # Patch CodecRobustDetector.__init__
    _orig_init = m.CodecRobustDetector.__init__.__wrapped__ if hasattr(m.CodecRobustDetector.__init__, '__wrapped__') else m.CodecRobustDetector.__init__

    def new_detector_init(self, embed_dim=256, n_codec_classes=10,
                          sample_rate=16000, n_attn_heads=4, use_temporal_attn=True):
        _orig_init(self, embed_dim, n_codec_classes, sample_rate, n_attn_heads)
        self.encoder.use_temporal_attn = use_temporal_attn

    import functools
    new_detector_init.__wrapped__ = _orig_init
    m.CodecRobustDetector.__init__ = new_detector_init

    print("Patches applied.")

apply_patches()

Patches applied.


In [10]:
import torch
from model import CodecRobustDetector
from data_utils import N_CODEC_CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def load_model(path, use_temporal_attn=True):
    model = CodecRobustDetector(embed_dim=256, n_codec_classes=N_CODEC_CLASSES,
                                use_temporal_attn=use_temporal_attn).to(device)
    state = torch.load(path, map_location=device)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    else:
        model.load_state_dict(state)
    model.eval()
    return model

checkpoint_map = {
    "1. Vanilla":         "/content/drive/MyDrive/CodecRobust/results/vanilla/best_model.pt",
    "2. Temporal":        "/content/drive/MyDrive/Applied-DL-2/trained_models/base_best_model.pt",
    "3. GRL+Contrastive": "/content/drive/MyDrive/CodecRobust/results/grl_contrastive_no_temporal/best_model.pt",
    "4. Full":            "/content/drive/MyDrive/Applied-DL-2/trained_models/full_best_model.pt",
}

vanilla    = load_model(checkpoint_map["1. Vanilla"],         use_temporal_attn=False)
temporal   = load_model(checkpoint_map["2. Temporal"],        use_temporal_attn=True)
grl_con    = load_model(checkpoint_map["3. GRL+Contrastive"], use_temporal_attn=False)
full_model = load_model(checkpoint_map["4. Full"],            use_temporal_attn=True)

print("All 4 models loaded.")

Device: cuda
All 4 models loaded.


In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from data_utils import apply_codec, codec_tag
from evaluate import compute_eer

def eval_key_codecs(models_dict, asv_root, processed_root, device, n_samples=500):
    import os

    # Only these 3 codecs
    key_codecs = [
        ("uncompressed", None),
        ("mp3", 32),
        ("opus", 16),
    ]

    proto_path = Path(asv_root) / 'LA' / 'ASVspoof2019_LA_cm_protocols' / 'ASVspoof2019.LA.cm.dev.trl.txt'
    df = pd.read_csv(proto_path, sep=' ', header=None,
                     names=['speaker','filename','dash','system_id','label'])
    available = set(f.replace('.pt','') for f in os.listdir(f'{processed_root}/uncompressed/dev/'))
    df = df[df['filename'].isin(available)].reset_index(drop=True)
    df = df.sample(min(n_samples, len(df)), random_state=42).reset_index(drop=True)

    results = {name: {} for name in models_dict}

    for codec, bitrate in key_codecs:
        tag = codec_tag(codec, bitrate)

        all_wfs, all_labs = [], []
        for _, row in df.iterrows():
            pt = Path(processed_root) / 'uncompressed' / 'dev' / f"{row['filename']}.pt"
            if not pt.exists():
                continue
            wav = torch.load(pt, weights_only=True).unsqueeze(0)
            wav = apply_codec(wav, 16000, codec, bitrate)
            all_wfs.append(wav.squeeze(0))
            all_labs.append(0 if row['label'] == 'bonafide' else 1)

        all_labs = np.array(all_labs)

        for name, model in models_dict.items():
            scores = []
            for i in range(0, len(all_wfs), 32):
                batch = torch.stack(all_wfs[i:i+32]).to(device)
                with torch.no_grad():
                    s = torch.softmax(model(batch)['deepfake_logits'], -1)[:, 1]
                scores.extend(s.cpu().numpy())
            eer = compute_eer(all_labs, np.array(scores))
            results[name][tag] = round(eer * 100, 1)

        line = f"{tag:<15}"
        for name in models_dict:
            line += f" {results[name][tag]:>12}%"
        print(line)

    return results

models_dict = {
    "Vanilla":  vanilla,
    "Temporal": temporal,
    "GRL+Con":  grl_con,
    "Full":     full_model,
}

header = f"{'Codec':<15}"
for name in models_dict:
    header += f" {name:>12}"
print(header)
print("-" * 65)

results = eval_key_codecs(models_dict, ASV_ROOT, PROCESSED_ROOT, device, n_samples=3000)

Codec                Vanilla     Temporal      GRL+Con         Full
-----------------------------------------------------------------
uncompressed             0.4%         12.6%          0.7%         15.4%
mp3_32                   0.2%         13.5%          1.5%         16.5%
opus_16                 98.4%         91.8%         97.9%         96.8%


In [12]:
import torch
from pathlib import Path

# Load one bonafide and one spoof sample
proto_path = Path(ASV_ROOT) / 'LA' / 'ASVspoof2019_LA_cm_protocols' / 'ASVspoof2019.LA.cm.dev.trl.txt'
import pandas as pd, os
df = pd.read_csv(proto_path, sep=' ', header=None,
                 names=['speaker','filename','dash','system_id','label'])
available = set(f.replace('.pt','') for f in os.listdir(f'{PROCESSED_ROOT}/uncompressed/dev/'))
df = df[df['filename'].isin(available)].reset_index(drop=True)

bonafide = df[df['label']=='bonafide'].iloc[0]['filename']
spoof    = df[df['label']=='spoof'].iloc[0]['filename']

for label, fname in [('bonafide', bonafide), ('spoof', spoof)]:
    pt = torch.load(f'{PROCESSED_ROOT}/uncompressed/dev/{fname}.pt', weights_only=True).unsqueeze(0).to(device)
    with torch.no_grad():
        out = vanilla(pt)
        prob = torch.softmax(out['deepfake_logits'], -1)
        print(f"Vanilla | {label:8s} | bonafide={prob[0,0]:.3f} spoof={prob[0,1]:.3f}")

Vanilla | bonafide | bonafide=0.999 spoof=0.001
Vanilla | spoof    | bonafide=0.003 spoof=0.997


In [15]:
!find /content -name "*.flac" 2>/dev/null | head -5
!find /content/drive/MyDrive -name "*.flac" 2>/dev/null | head -5

^C
^C


In [18]:
import numpy as np
import pandas as pd

# Latest validated results
results = {
    'Vanilla': {
        'uncompressed': 21.22, 'aac_32': 24.07, 'aac_64': 21.66, 'aac_128': 21.05,
        'opus_16': 72.97, 'opus_32': 81.84, 'opus_64': 66.54
    },
    '+Temporal': {
        'uncompressed': 28.28, 'aac_32': 26.74, 'aac_64': 26.23, 'aac_128': 25.61,
        'opus_16': 73.65, 'opus_32': 52.48, 'opus_64': 43.55
    },
    '+Temporal+GRL+Con': {
        'uncompressed': 25.38, 'aac_32': 23.91, 'aac_64': 25.80, 'aac_128': 25.87,
        'opus_16': 60.69, 'opus_32': 46.10, 'opus_64': 44.80
    }
}

DAILY_CALLS = 10000
FAKE_RATE   = 0.05
REAL_RATE   = 0.95
daily_fakes = DAILY_CALLS * FAKE_RATE
daily_real  = DAILY_CALLS * REAL_RATE

print(f"Assumptions: {DAILY_CALLS:,} calls/day | {FAKE_RATE*100:.0f}% fake | {REAL_RATE*100:.0f}% real")
print()

# Metric 1: Fraud slips through per day
print("=" * 70)
print("OUT OF 100 FAKE CALLS, HOW MANY SLIP THROUGH?")
print("=" * 70)
print(f"{'Codec':<15}", end="")
for m in results: print(f"{m:>20}", end="")
print()
print("-" * 70)
for codec in results['Vanilla']:
    print(f"{codec:<15}", end="")
    for m in results:
        eer = results[m][codec]
        print(f"{eer:>19.1f}%", end="")
    print()

# Metric 2: Daily fraud at scale
print("\n" + "=" * 70)
print(f"DAILY FRAUD EXPOSURE ({DAILY_CALLS:,} CALLS/DAY, {FAKE_RATE*100:.0f}% FAKE)")
print("=" * 70)
print(f"{'Codec':<15}", end="")
for m in results: print(f"{m:>20}", end="")
print()
print("-" * 70)
for codec in results['Vanilla']:
    print(f"{codec:<15}", end="")
    for m in results:
        fraud = round(daily_fakes * results[m][codec] / 100)
        print(f"{fraud:>19} calls", end="")
    print()

# Metric 3: Customers wrongly blocked per day
print("\n" + "=" * 70)
print(f"REAL CUSTOMERS WRONGLY BLOCKED PER DAY")
print("=" * 70)
print(f"{'Codec':<15}", end="")
for m in results: print(f"{m:>20}", end="")
print()
print("-" * 70)
for codec in results['Vanilla']:
    print(f"{codec:<15}", end="")
    for m in results:
        blocked = round(daily_real * results[m][codec] / 100)
        print(f"{blocked:>19} people", end="")
    print()

# Metric 4: Annual fraud prevented by full model vs vanilla
print("\n" + "=" * 70)
print("ANNUAL FRAUD CALLS PREVENTED: +Temporal+GRL+Con vs Vanilla")
print("=" * 70)
for codec in results['Vanilla']:
    v = results['Vanilla'][codec]
    f = results['+Temporal+GRL+Con'][codec]
    daily_saved = round(daily_fakes * (v - f) / 100)
    annual_saved = daily_saved * 365
    direction = "prevented" if daily_saved > 0 else "MORE slipping through"
    print(f"  {codec:<15}: {abs(annual_saved):>6,} calls/year {direction}")

# Metric 5: Deployment verdict
print("\n" + "=" * 70)
print("DEPLOYMENT VERDICT (EER > 30% = undeployable for that codec)")
print("=" * 70)
for m in results:
    undeployable = [c for c, e in results[m].items() if e > 30]
    deployable   = [c for c, e in results[m].items() if e <= 30]
    print(f"  {m}:")
    print(f"    Deployable on   : {', '.join(deployable) if deployable else 'none'}")
    print(f"    Undeployable on : {', '.join(undeployable) if undeployable else 'none'}")

Assumptions: 10,000 calls/day | 5% fake | 95% real

OUT OF 100 FAKE CALLS, HOW MANY SLIP THROUGH?
Codec                       Vanilla           +Temporal   +Temporal+GRL+Con
----------------------------------------------------------------------
uncompressed                  21.2%               28.3%               25.4%
aac_32                        24.1%               26.7%               23.9%
aac_64                        21.7%               26.2%               25.8%
aac_128                       21.1%               25.6%               25.9%
opus_16                       73.0%               73.7%               60.7%
opus_32                       81.8%               52.5%               46.1%
opus_64                       66.5%               43.5%               44.8%

DAILY FRAUD EXPOSURE (10,000 CALLS/DAY, 5% FAKE)
Codec                       Vanilla           +Temporal   +Temporal+GRL+Con
----------------------------------------------------------------------
uncompressed              

In [19]:
# Metric 6: Breaking point
print("=" * 70)
print("AT WHAT COMPRESSION DOES EACH MODEL START FAILING?")
print("(failing = more than 1 in 3 calls wrong, EER > 33%)")
print("=" * 70)
for m in results:
    failing = [c for c, e in results[m].items() if e > 33]
    ok      = [c for c, e in results[m].items() if e <= 33]
    print(f"  {m}:")
    print(f"    Handles : {', '.join(ok) if ok else 'none'}")
    print(f"    Breaks  : {', '.join(failing) if failing else 'none'}")

# Metric 7: Consistency
print("\n" + "=" * 70)
print("HOW CONSISTENT IS EACH MODEL? (lower std = more predictable)")
print("=" * 70)
for m in results:
    eers = list(results[m].values())
    print(f"  {m}:")
    print(f"    Best    : {min(eers):.1f}% on {min(results[m], key=results[m].get)}")
    print(f"    Worst   : {max(eers):.1f}% on {max(results[m], key=results[m].get)}")
    print(f"    Std Dev : {np.std(eers):.1f}% — {'unreliable' if np.std(eers) > 20 else 'consistent'}")

# Metric 8: Recovery rate
print("\n" + "=" * 70)
print("HOW MUCH DID FULL MODEL RECOVER VS VANILLA?")
print("(positive = improvement, negative = got worse)")
print("=" * 70)
for codec in results['Vanilla']:
    v = results['Vanilla'][codec]
    f = results['+Temporal+GRL+Con'][codec]
    recovery = v - f
    direction = "better" if recovery > 0 else "worse"
    print(f"  {codec:<15}: {recovery:+.2f}% {direction}")

# Metric 9: Which model wins per codec
print("\n" + "=" * 70)
print("WHICH MODEL IS BEST PER CODEC?")
print("=" * 70)
for codec in results['Vanilla']:
    best_model = min(results, key=lambda m: results[m][codec])
    best_eer   = results[best_model][codec]
    print(f"  {codec:<15}: {best_model} ({best_eer:.1f}%)")

# Metric 10: If you had to pick one model to deploy
print("\n" + "=" * 70)
print("IF YOU HAD TO PICK ONE MODEL TO DEPLOY ON A PHONE NETWORK:")
print("=" * 70)
for m in results:
    eers     = list(results[m].values())
    avg      = np.mean(eers)
    worst    = max(eers)
    n_above50 = sum(1 for e in eers if e > 50)
    print(f"  {m}:")
    print(f"    Average EER across all codecs : {avg:.1f}%")
    print(f"    Worst case EER                : {worst:.1f}%")
    print(f"    Codecs with EER > 50%         : {n_above50}")

AT WHAT COMPRESSION DOES EACH MODEL START FAILING?
(failing = more than 1 in 3 calls wrong, EER > 33%)
  Vanilla:
    Handles : uncompressed, aac_32, aac_64, aac_128
    Breaks  : opus_16, opus_32, opus_64
  +Temporal:
    Handles : uncompressed, aac_32, aac_64, aac_128
    Breaks  : opus_16, opus_32, opus_64
  +Temporal+GRL+Con:
    Handles : uncompressed, aac_32, aac_64, aac_128
    Breaks  : opus_16, opus_32, opus_64

HOW CONSISTENT IS EACH MODEL? (lower std = more predictable)
  Vanilla:
    Best    : 21.1% on aac_128
    Worst   : 81.8% on opus_32
    Std Dev : 26.0% — unreliable
  +Temporal:
    Best    : 25.6% on aac_128
    Worst   : 73.7% on opus_16
    Std Dev : 16.9% — consistent
  +Temporal+GRL+Con:
    Best    : 23.9% on aac_32
    Worst   : 60.7% on opus_16
    Std Dev : 13.4% — consistent

HOW MUCH DID FULL MODEL RECOVER VS VANILLA?
(positive = improvement, negative = got worse)
  uncompressed   : -4.16% worse
  aac_32         : +0.16% better
  aac_64         : -4.14% wo